In [11]:
#importando libs
import os

import random
import numpy as np
import keras

import matplotlib.pyplot as plt
from matplotlib.pyplot import imshow

from keras.preprocessing import image
from keras.applications.imagenet_utils import preprocess_input
from keras.layers import Dense
from keras.models import Model
from keras.applications import VGG16



In [25]:
import kagglehub
import os
import shutil

# ==========================================
# 1. Verificar se o dataset final já existe
# ==========================================

dataset_final = "./dataset"

healthy_dir = os.path.join(dataset_final, "Healthy")
diseased_dir = os.path.join(dataset_final, "Sick")


if os.path.exists(healthy_dir) and os.path.exists(diseased_dir):

    healthy_count = len(os.listdir(healthy_dir))
    sick_count = len(os.listdir(diseased_dir))

    if healthy_count > 0 and sick_count > 0:

        print("Dataset já está organizado.")
        print("Não é necessário baixar novamente.")
        print("Healthy:", healthy_count)
        print("Sick:", sick_count)

    else:
        print("As pastas existem, mas estão vazias.")
        print("Será necessário organizar o dataset.")

        # Continua para o download/organização
        path = kagglehub.dataset_download(
            "vaishaligbhujade/soybean-leaf-dataset-for-disease-classification"
        )

else:

    print("Dataset ainda não foi organizado.")
    print("Verificando/baixando dataset original...")

    path = kagglehub.dataset_download(
        "vaishaligbhujade/soybean-leaf-dataset-for-disease-classification"
    )

    print("Dataset original:", path)

    # Criar pastas finais
    os.makedirs(healthy_dir, exist_ok=True)
    os.makedirs(diseased_dir, exist_ok=True)

    # ==========================================
    # Categorias
    # ==========================================

    healthy = "Healty"

    diseases = [
        "Bacterial Pustule",
        "Frogeye Leaf Spot",
        "Rust",
        "Sudden Death Syndrome",
        "Target Leaf Spot",
        "Yellow Mosaic"
    ]

    # ==========================================
    # Copiar Healthy
    # ==========================================

    origem_healthy = os.path.join(path, healthy)

    for arquivo in os.listdir(origem_healthy):

        origem = os.path.join(origem_healthy, arquivo)

        if os.path.isfile(origem):

            destino = os.path.join(healthy_dir, arquivo)

            shutil.copy2(origem, destino)

    # ==========================================
    # Copiar doenças
    # ==========================================

    contador = 0

    for disease in diseases:

        origem_disease = os.path.join(path, disease)

        if not os.path.exists(origem_disease):
            print(f"Pasta não encontrada: {disease}")
            continue

        for arquivo in os.listdir(origem_disease):

            origem = os.path.join(origem_disease, arquivo)

            if os.path.isfile(origem):

                extensao = os.path.splitext(arquivo)[1]

                novo_nome = f"diseased_{contador}{extensao}"

                destino = os.path.join(
                    diseased_dir,
                    novo_nome
                )

                shutil.copy2(origem, destino)

                contador += 1




Dataset já está organizado.
Não é necessário baixar novamente.
Healthy: 110
Sick: 660


In [13]:
#Pegando as imagens

dataset_dir = r"dataset"

healthy_dir = os.path.join(dataset_dir, "Healthy")
sick_dir = os.path.join(dataset_dir, "Sick")

data = []


# ==========================================
# Healthy = 0
# ==========================================

for root, dirs, files in os.walk(healthy_dir):

    for file in files:

        if file.lower().endswith((".jpg", ".jpeg", ".png")):

            path = os.path.join(root, file)

            img = image.load_img(
                path,
                target_size=(224, 224)
            )

            x = image.img_to_array(img)
            x = np.expand_dims(x, axis=0)
            x = preprocess_input(x)

            data.append({
                "x": x[0],
                "y": 0
            })


# ==========================================
# Sick = 1
# ==========================================

for root, dirs, files in os.walk(sick_dir):

    for file in files:

        if file.lower().endswith((".jpg", ".jpeg", ".png")):

            path = os.path.join(root, file)

            img = image.load_img(
                path,
                target_size=(224, 224)
            )

            x = image.img_to_array(img)
            x = np.expand_dims(x, axis=0)
            x = preprocess_input(x)

            data.append({
                "x": x[0],
                "y": 1
            })



print("Total de imagens:", len(data))


Total de imagens: 770


In [14]:
random.shuffle(data)

#proporção recomendada
    #70% → treino
    #15% → validação
    #15% → teste

#Valor do treino 70% dos dados 
train_split = 0.70

#Valor da validação 15% dos dados
val_split = 0.15

#pegando o valor das imagens treinada (o indice onde o treino termina)
idx_val = int(train_split * len(data))

#pegando o valor onde termina validação (o indice onde a validação termina)
idx_test = int((train_split + val_split) * len(data))

#pega do inicio até o valor da validação
train = data[:idx_val]

#pega do valor da validação até o índice do teste
val = data[idx_val:idx_test]

#pega do indice do teste até o final
test = data[idx_test:]

In [15]:
#Separando assim o modelo vai aprender aprender a relação onde temos todas as imagens e a resposta correta para cada imagem , sendo
# x = imagens
# y = rótulos (0 = Healthy, 1 = Sick)
# As list comprehensions percorrem cada elemento e extraem "x" ou "y".

#imagens separadas para treino , separamos as imagens e os rotulos
x_train = np.array([t["x"] for t in train])
y_train = np.array([t["y"] for t in train])

#imagens separadas para treino , separamos as imagens e os rotulos
x_val = np.array([t["x"] for t in val])
y_val = np.array([t["y"] for t in val])

#imagens separadas para treino , separamos as imagens e os rotulos
x_test = np.array([t["x"] for t in test])
y_test = np.array([t["y"] for t in test])

In [16]:
# Carregamos a VGG16 já pré-treinada no ImageNet.
# Vamos aproveitar o conhecimento aprendido pela rede e adaptá-la
# para o nosso problema de classificação
vgg = VGG16(
    weights="imagenet",
    include_top=True
)

In [17]:
#Vamos usar a mesma entrada que a Vgg espera "224 × 224 "
inp = vgg.input

#Criando um neuronio para classificação da imagem , onde indicamos o numero 1 pois são duas classificações possíveis 
#Doente ou saudável
new_classification_layer = Dense(
    1,
    activation="sigmoid"
)

#indicamos a penúltima camada , pois última e do treinamengo original da Vgg
out = new_classification_layer(
    vgg.layers[-2].output
)

# Cria o novo modelo usando a entrada da VGG e a nova saída
model = Model(inp, out)

In [18]:
for layer in model.layers[:-1]:
    layer.trainable = False

model.layers[-1].trainable = True

In [19]:
#Dizemos ao modelo como ele deve aprender 
#adam modelo que calcula a loss para ajustar os pessos 
#metrics, foi usado para acompanhar a accuracy
model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

In [20]:
# Treinando o modelo com as imagens e os respectivos rótulos de treino.
# batch_size=32: o modelo processa 32 imagens por vez.
# epochs=10: o treinamento percorre todo o conjunto de treino 10 vezes.
# validation_data: utiliza as imagens de validação para acompanhar o desempenho
# do modelo durante o treinamento.
# history: armazena os resultados de cada época, como loss e accuracy.

history = model.fit(
    x_train,
    y_train,
    batch_size=32,
    epochs=10,
    validation_data=(x_val, y_val)
)

Epoch 1/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 28s 2s/step - accuracy: 0.7774 - loss: 0.6567 - val_accuracy: 0.9652 - val_loss: 0.1089
Epoch 2/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 27s 2s/step - accuracy: 0.9518 - loss: 0.1533 - val_accuracy: 0.9826 - val_loss: 0.0569
Epoch 3/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 27s 2s/step - accuracy: 0.9685 - loss: 0.0727 - val_accuracy: 0.9913 - val_loss: 0.0357
Epoch 4/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 27s 2s/step - accuracy: 0.9870 - loss: 0.0517 - val_accuracy: 1.0000 - val_loss: 0.0302
Epoch 5/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 27s 2s/step - accuracy: 0.9889 - loss: 0.0426 - val_accuracy: 1.0000 - val_loss: 0.0273
Epoch 6/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 28s 2s/step - accuracy: 0.9907 - loss: 0.0369 - val_accuracy: 1.0000 - val_loss: 0.0252
Epoch 7/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 29s 2s/step - accuracy: 0.9907 - loss: 0.0322 - val_accuracy: 1.0000 - val_loss: 0.0236
Epoch 8/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 30s 2s/step - accuracy: 0.9907 - loss: 0.0286 - val_accuracy: 1.0000 - val_loss:

### Resultados do treinamento

Durante o treinamento, o modelo foi executado por 10 épocas. Em cada época,
foram acompanhados os valores de accuracy e loss tanto no conjunto de treino
quanto no conjunto de validação.

- **accuracy:** percentual de acertos no conjunto de treino
- **loss:** erro do modelo no conjunto de treino
- **val_accuracy:** percentual de acertos no conjunto de validação
- **val_loss:** erro do modelo no conjunto de validação

Ao final da 10ª época, os resultados no conjunto de treino foram:

- **Accuracy:** 99,63%
- **Loss:** 0,022

Isso significa que, ao final do treinamento, o modelo classificou corretamente
99,81% das imagens do conjunto de treino, apresentando uma loss de 0,0214.

In [21]:
#aqui estamos usando nossos dados de teste que foram dividos no dataset antes 
#15 % teste 
loss, accuracy = model.evaluate(
    x_test,
    y_test,
    verbose=0
)

# Resultado da avaliação final do modelo no conjunto de teste
print("Test loss:", loss)
print("Test accuracy:", accuracy)


# Quantidade de imagens em cada conjunto
print("Total:", len(data))
print("Treino:", len(train))
print("Validação:", len(val))
print("Teste:", len(test))

# Quantidade de imagens de cada classe
# 0 = Healthy | 1 = Sick
# np.sum função simple do numpy
print("Healthy treino:", np.sum(y_train == 0))
print("Sick treino:", np.sum(y_train == 1))
print("Healthy teste:", np.sum(y_test == 0))
print("Sick teste:", np.sum(y_test == 1))

Test loss: 0.08763816952705383
Test accuracy: 0.9655172228813171
Total: 770
Treino: 539
Validação: 115
Teste: 116
Healthy treino: 82
Sick treino: 457
Healthy teste: 17
Sick teste: 99


In [ ]:

# teste simples com duas imagens de exmeplo , baixadas aleatóriamente da internet 
# tendo como objetivo um simples teste

# Caminho da imagem
img_path = r"folha_teste_d.jpg"

# Carrega e redimensiona para ficar de acordo com as dimensões do modelo 
img = image.load_img(img_path, target_size=(224, 224))

# Converte para array
x = image.img_to_array(img)


x = np.expand_dims(x, axis=0)


x = preprocess_input(x)

# Faz a previsão
probabilidade = model.predict(x)[0][0]

#arredondando
print("Probabilidade de estar doente:", probabilidade.round(3))

#definir em 0.8 a margem 
if probabilidade >= 0.8:
    print("Planta DOENTE")
else:
    print("Planta SAUDÁVEL")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 215ms/step
Probabilidade de estar doente: 0.999
Planta DOENTE
